BƯỚC 1: TẢI TÀI NGUYÊN (FASTTEXT)

In [1]:
# [BƯỚC 1] TẢI VỀ FASTTEXT PRE-TRAINED VECTORS (1.5GB)
import os

# Kiểm tra: Nếu chưa có file vector thì mới tải
if not os.path.exists('crawl-300d-2M.vec'):
    print("⏳ Đang tải FastText từ Facebook (Vui lòng chờ 1-2 phút)...")
    # Tải file zip
    !wget -q https://dl.fbaipublicfiles.com/fasttext/vectors-english/crawl-300d-2M.vec.zip

    print("⏳ Đang giải nén (Unzipping)...")
    # Giải nén
    !unzip -q crawl-300d-2M.vec.zip

    print("✅ Đã tải và giải nén xong! File 'crawl-300d-2M.vec' đã sẵn sàng.")
else:
    print("✅ File FastText đã có sẵn, không cần tải lại!")

⏳ Đang tải FastText từ Facebook (Vui lòng chờ 1-2 phút)...
⏳ Đang giải nén (Unzipping)...
✅ Đã tải và giải nén xong! File 'crawl-300d-2M.vec' đã sẵn sàng.


BƯỚC 2: XỬ LÝ DỮ LIỆU & SỐ HÓA (TOKENIZATION)

In [2]:
# [BƯỚC 2] LOAD DATA & TOKENIZATION
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# --- CẤU HÌNH ---
MAX_WORDS = 20000       # Chỉ lấy 20k từ phổ biến nhất
MAX_LEN = 250           # Độ dài cố định mỗi câu
EMBEDDING_DIM = 300     # FastText luôn là 300 chiều

# 1. Load & Clean Data
print("⏳ Đang xử lý dữ liệu...")
# Tự động tìm file csv
data_path = 'MBTI 500.csv'
if not os.path.exists(data_path):
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            if '500' in filename: data_path = os.path.join(dirname, filename); break

try:
    df = pd.read_csv(data_path)
    df.dropna(subset=['posts', 'type'], inplace=True)

    # Hàm clean
    def basic_clean(text):
        text = str(text).lower()
        text = re.sub(r'http\S+|www\.\S+', '', text)
        # Xóa 16 từ khóa MBTI (Chống nhìn bài)
        mbti_types = ['infj', 'intp', 'infp', 'entp', 'intj', 'istj', 'entj', 'istp',
                      'enfj', 'enfp', 'isfj', 'esfj', 'isfp', 'estj', 'estp', 'esfp']
        pattern = r'\b(?:' + '|'.join(mbti_types) + r')\b'
        text = re.sub(pattern, '', text)
        return re.sub(r'[^a-z0-9\s]', '', text).strip()

    df['clean_text'] = df['posts'].apply(basic_clean)
    print(f"   -> Đã load {len(df)} dòng dữ liệu.")

    # 2. Tokenizer (Học từ vựng)
    print("⏳ Đang số hóa văn bản (Tokenizing)...")
    tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
    tokenizer.fit_on_texts(df['clean_text'])

    # Chuyển chữ thành số
    sequences = tokenizer.texts_to_sequences(df['clean_text'])

    # Padding (Đệm cho bằng nhau)
    X_data = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

    word_index = tokenizer.word_index
    print(f"✅ Dữ liệu đầu vào (X) đã xong! Kích thước: {X_data.shape}")

except Exception as e:
    print(f"❌ LỖI: {e}. Hãy kiểm tra lại file csv!")

⏳ Đang xử lý dữ liệu...
   -> Đã load 106067 dòng dữ liệu.
⏳ Đang số hóa văn bản (Tokenizing)...
✅ Dữ liệu đầu vào (X) đã xong! Kích thước: (106067, 250)


BƯỚC 3: TẠO MA TRẬN EMBEDDING

In [3]:
# [BƯỚC 3] TRÍCH XUẤT VECTOR TỪ FASTTEXT
print("⏳ Đang quét file FastText để lấy vector (Mất khoảng 1-2 phút)...")

embeddings_index = {}
# Mở file và đọc từng dòng (Streaming)
with open('crawl-300d-2M.vec', encoding='utf8') as f:
    for line in f:
        values = line.rstrip().rsplit(' ')
        word = values[0]
        # Chỉ lấy vector nếu từ đó nằm trong từ điển của mình
        if word in word_index:
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs

print(f"   -> Đã tìm thấy {len(embeddings_index)} vector từ trùng khớp.")

# Tạo ma trận khởi tạo (Init Matrix)
num_words = min(MAX_WORDS, len(word_index) + 1)
embedding_matrix = np.zeros((num_words, EMBEDDING_DIM))

for word, i in word_index.items():
    if i >= MAX_WORDS: continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        # Gán vector FastText vào đúng vị trí
        embedding_matrix[i] = embedding_vector
    else:
        # Từ nào FastText không có thì để vector 0 (hoặc random)
        pass

print(f"✅ Ma trận Embedding đã sẵn sàng! Kích thước: {embedding_matrix.shape}")

⏳ Đang quét file FastText để lấy vector (Mất khoảng 1-2 phút)...
   -> Đã tìm thấy 119787 vector từ trùng khớp.
✅ Ma trận Embedding đã sẵn sàng! Kích thước: (20000, 300)


BƯỚC 4: HUẤN LUYỆN TEXT-CNN & BÁO CÁO

In [4]:
# [BƯỚC 4] DỰNG MODEL & HUẤN LUYỆN
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Concatenate, Dropout
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score

# --- A. HÀM DỰNG MODEL ---
def build_fasttext_cnn():
    # 1. Input
    inputs = Input(shape=(MAX_LEN,), dtype='int32')

    # 2. Embedding (Nạp ma trận FastText vừa tạo vào đây)
    embedding_layer = Embedding(
        input_dim=num_words,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix], # <--- KEY POINT
        input_length=MAX_LEN,
        trainable=False             # <--- Đóng băng để giữ kiến thức FastText
    )(inputs)

    # 3. Convolutional Filters (3 nhánh)
    # Nhánh 1: Quét 3 từ
    conv_0 = Conv1D(filters=128, kernel_size=3, activation='relu')(embedding_layer)
    pool_0 = GlobalMaxPooling1D()(conv_0)

    # Nhánh 2: Quét 4 từ
    conv_1 = Conv1D(filters=128, kernel_size=4, activation='relu')(embedding_layer)
    pool_1 = GlobalMaxPooling1D()(conv_1)

    # Nhánh 3: Quét 5 từ
    conv_2 = Conv1D(filters=128, kernel_size=5, activation='relu')(embedding_layer)
    pool_2 = GlobalMaxPooling1D()(conv_2)

    # 4. Gộp & Phân loại
    merged = Concatenate()([pool_0, pool_1, pool_2])
    dense = Dense(128, activation='relu')(merged)
    dropout = Dropout(0.5)(dense) # Chống Overfitting
    outputs = Dense(1, activation='sigmoid')(dropout)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# --- B. TRAINING LOOP ---
axes = {'IE': ('I', 'E'), 'NS': ('N', 'S'), 'TF': ('F', 'T'), 'JP': ('J', 'P')}
results = []

print("🚀 BẮT ĐẦU HUẤN LUYỆN CHIẾN DỊCH TEXT-CNN...")
print("=" * 70)

for axis, (char0, char1) in axes.items():
    print(f"\n🔥 ĐANG TRAIN TRỤC: {axis} ({char0} vs {char1})")

    # Chuẩn bị nhãn
    y_data = df['type'].apply(lambda x: 1 if char1 in x else 0).values

    # Chia tập (Train 90% - Val 10%)
    X_train, X_val, y_train, y_val = train_test_split(
        X_data, y_data, test_size=0.1, random_state=42, stratify=y_data
    )

    # Trọng số lớp (Xử lý mất cân bằng)
    weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    class_weights = {0: weights[0], 1: weights[1]}

    # Reset RAM & Build Model
    tf.keras.backend.clear_session()
    model = build_fasttext_cnn()

    # Early Stopping (Dừng nếu không khá hơn sau 3 epoch)
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    # Train
    print("   -> Training (Epochs=10, Batch=64)...")
    history = model.fit(X_train, y_train,
                        validation_data=(X_val, y_val),
                        epochs=10,
                        batch_size=64,
                        class_weight=class_weights,
                        callbacks=[early_stop],
                        verbose=1)

    # Đánh giá
    y_prob = model.predict(X_val)
    y_pred = (y_prob > 0.5).astype(int)

    acc = accuracy_score(y_val, y_pred)
    f1_target = f1_score(y_val, y_pred, pos_label=1)
    mf1 = f1_score(y_val, y_pred, average='macro')
    wf1 = f1_score(y_val, y_pred, average='weighted')

    results.append({'Trục': axis, 'Accuracy': acc, 'F1-Target': f1_target, 'Macro F1': mf1, 'Weighted F1': wf1})
    print(f"   ✅ Xong {axis}! Acc: {acc:.4f} | Macro F1: {mf1:.4f}")

# --- C. HIỂN THỊ KẾT QUẢ ---
print("\n" + "="*70)
print("🏆 BẢNG KẾT QUẢ TEXT-CNN + FASTTEXT (MBTI 500)")
print("="*70)
final_df = pd.DataFrame(results)
pd.options.display.float_format = '{:.4f}'.format
print(final_df)

🚀 BẮT ĐẦU HUẤN LUYỆN CHIẾN DỊCH TEXT-CNN...

🔥 ĐANG TRAIN TRỤC: IE (I vs E)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


   -> Training (Epochs=10, Batch=64)...
Epoch 1/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 26s 13ms/step - accuracy: 0.5775 - loss: 0.6670 - val_accuracy: 0.7294 - val_loss: 0.5969
Epoch 2/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 16s 11ms/step - accuracy: 0.7578 - loss: 0.5394 - val_accuracy: 0.7900 - val_loss: 0.4813
Epoch 3/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 16s 11ms/step - accuracy: 0.8181 - loss: 0.4394 - val_accuracy: 0.7186 - val_loss: 0.5862
Epoch 4/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.8803 - loss: 0.3199 - val_accuracy: 0.7902 - val_loss: 0.5113
Epoch 5/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.9317 - loss: 0.2029 - val_accuracy: 0.7617 - val_loss: 0.6145
332/332 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step
   ✅ Xong IE! Acc: 0.7900 | Macro F1: 0.7198

🔥 ĐANG TRAIN TRỤC: NS (N vs S)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


   -> Training (Epochs=10, Batch=64)...
Epoch 1/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 24s 13ms/step - accuracy: 0.6456 - loss: 0.6306 - val_accuracy: 0.8139 - val_loss: 0.5010
Epoch 2/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - accuracy: 0.8110 - loss: 0.4607 - val_accuracy: 0.8509 - val_loss: 0.4038
Epoch 3/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - accuracy: 0.8805 - loss: 0.3140 - val_accuracy: 0.8813 - val_loss: 0.3037
Epoch 4/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - accuracy: 0.9359 - loss: 0.1777 - val_accuracy: 0.8899 - val_loss: 0.2939
Epoch 5/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - accuracy: 0.9610 - loss: 0.1081 - val_accuracy: 0.8981 - val_loss: 0.3352
Epoch 6/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - accuracy: 0.9696 - loss: 0.0830 - val_accuracy: 0.8667 - val_loss: 0.4502
Epoch 7/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - accuracy: 0.9782 - loss: 0.0615 - val_accuracy: 0.8598 - val_loss: 0.4668
332/332 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


   -> Training (Epochs=10, Batch=64)...
Epoch 1/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 23s 13ms/step - accuracy: 0.7043 - loss: 0.5629 - val_accuracy: 0.8238 - val_loss: 0.4052
Epoch 2/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - accuracy: 0.8417 - loss: 0.3799 - val_accuracy: 0.7623 - val_loss: 0.5137
Epoch 3/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - accuracy: 0.8886 - loss: 0.2841 - val_accuracy: 0.8280 - val_loss: 0.3957
Epoch 4/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - accuracy: 0.9347 - loss: 0.1788 - val_accuracy: 0.8286 - val_loss: 0.4365
Epoch 5/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - accuracy: 0.9628 - loss: 0.1034 - val_accuracy: 0.8264 - val_loss: 0.5478
Epoch 6/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - accuracy: 0.9718 - loss: 0.0798 - val_accuracy: 0.8196 - val_loss: 0.7255
332/332 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
   ✅ Xong TF! Acc: 0.8280 | Macro F1: 0.8146

🔥 ĐANG TRAIN TRỤC: JP (J vs P)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


   -> Training (Epochs=10, Batch=64)...
Epoch 1/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - accuracy: 0.5632 - loss: 0.6803 - val_accuracy: 0.6567 - val_loss: 0.6105
Epoch 2/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - accuracy: 0.7020 - loss: 0.5780 - val_accuracy: 0.7008 - val_loss: 0.5660
Epoch 3/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - accuracy: 0.7650 - loss: 0.4937 - val_accuracy: 0.6957 - val_loss: 0.5726
Epoch 4/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.8459 - loss: 0.3671 - val_accuracy: 0.6780 - val_loss: 0.6917
Epoch 5/10
1492/1492 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - accuracy: 0.9066 - loss: 0.2410 - val_accuracy: 0.6897 - val_loss: 0.8671
332/332 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step
   ✅ Xong JP! Acc: 0.7008 | Macro F1: 0.6965

🏆 BẢNG KẾT QUẢ TEXT-CNN + FASTTEXT (MBTI 500)
  Trục  Accuracy  F1-Target  Macro F1  Weighted F1
0   IE    0.7900     0.5794    0.7198       0.7929
1   NS    0.8899     0.4823    0.7103       0.8988
2   TF    0.82